# Module 1: Potential Outcomes, Estimands, and What You Are Actually Estimating

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Before any method, a question: **which number is the target?**

There are several quantities that could all reasonably be called "the effect
of the program", they are different numbers, and a given design reaches some
and not others. Reporting one while describing another is the most common
failure in applied causal work, and it happens silently.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. The estimands

For unit *i* write **Y_i(1)** and **Y_i(0)** for the outcome under treatment
and under control. Only one is observed. Averaging the difference over
different populations gives different targets.

| Estimand | Definition | Answers |
|---|---|---|
| **ATE** | E[Y(1) − Y(0)] over all units | would it help if everyone took it |
| **ATT** | E[Y(1) − Y(0) \| D = 1] | did it help those who took it |
| **ATU** | E[Y(1) − Y(0) \| D = 0] | would it have helped the others |
| **LATE** | the effect for units an instrument moves | did it help the compliers |

A difference in differences identifies the **ATT** and nothing else. The ATE
and ATU require an assumption about units for which no treated observation
exists anywhere.

**And "the ATT" is still not one number**, because averaging over agencies
and averaging over incidents are different operations.

In [ ]:
pooled = fit(d, KEEP)
per = {a: fit(d[d["agency_id"].isin([a] + COMPARISON)], [a]) for a in KEEP}

rows = [{"agency": NAME[a], "estimate": f"{per[a][0]:+.2f}%",
         "95 percent interval": f"[{per[a][1]:+.1f}, {per[a][2]:+.1f}]"}
        for a in KEEP]
rows.append({"agency": "ATT, agencies weighted equally",
             "estimate": f"{np.mean([per[a][0] for a in KEEP]):+.2f}%",
             "95 percent interval": ""})
rows.append({"agency": "ATT, weighted by incidents",
             "estimate": f"{pooled[0]:+.2f}%",
             "95 percent interval": f"[{pooled[1]:+.1f}, {pooled[2]:+.1f}]"})
rows.append({"agency": "THE TRUTH", "estimate": f"{TRUTH:+.2f}%",
             "95 percent interval": ""})
pd.DataFrame(rows).set_index("agency")

**Three percentage points apart, and both are the ATT.**

The pooled Poisson weights each agency by the incidents it contributes, so
Stonewick's 57 incidents a month count for more than Pinecrest's 3.6.
Averaging the four agency level estimates weights each agency equally.

Neither is wrong. They answer different questions:

- **Weighted by incidents:** what happened to a typical *incident*. This is
  the quantity a statewide total responds to.
- **Weighted by agency:** what happened at a typical *agency*. This is the
  quantity a chief deciding whether to adopt the program cares about.

The planted effect is identical at every agency, so the truth is the same for
both, and the three point gap is entirely sampling noise amplified by giving
the small noisy agencies equal weight.

**Say which one you report.** In a study with heterogeneous effects they can
differ by far more than three points and in either direction.

## 3. Why the ATE is not available

The ATE requires E[Y(1)] for the comparison agencies, and nothing in these
data speaks to it. That is not a gap that more careful estimation fills.

In [ ]:
print("  the selection rule, in one table\n")
for a in sorted(BASELINE.index, key=lambda x: -BASELINE[x]):
    mark = "  TREATED" if a in TRAINED else ""
    print(f"    {BASELINE[a]:.3f}   {NAME[a]:34s}{mark}")
print(f"\n  treated mean baseline rate:  {BASELINE[TRAINED].mean():.3f}")
print(f"  untreated mean:              "
      f"{BASELINE[[a for a in BASELINE.index if a not in TRAINED]].mean():.3f}")

Four of the top five were selected. **There is no untreated agency at the top
of the distribution and no treated agency at the bottom**, so the data
contain no information about what the program does to a low rate agency.

Claiming an ATE here requires assuming the effect is constant across the
distribution. That may be reasonable and it is an **assumption**, not a
finding, and it should appear in the text as one. Advanced
[Module 10](Module_10_Propensity_Scores.ipynb) measures the lack of overlap
directly.

## 4. What the estimand becomes under heterogeneity

If the effect really is constant, the choice of weights does not matter.
Simulate a world where it is not, and watch the two estimands separate.

In [ ]:
rng = np.random.default_rng(3)
# give the two large agencies a 20 percent effect and the two small ones 4 percent
effects = {"A001": 0.20, "A002": 0.20, "A004": 0.04, "A010": 0.04}

s = d.copy()
mult = np.ones(len(s))
for a, e in effects.items():
    sel = (s["agency_id"] == a) & (s["period"] == "after")
    mult[sel.values] = (1 - e) / 0.88          # replace the planted 12 percent
s["y"] = rng.poisson(np.maximum(s["n_uof"].values * mult, 0.01))

pooled_h = fit(s, KEEP, outcome="y")[0]
per_h = [fit(s[s["agency_id"].isin([a] + COMPARISON)], [a], outcome="y")[0]
         for a in KEEP]
print("  a world where the effect is 20 percent at the two large agencies")
print("  and 4 percent at the two small ones\n")
print(f"    ATT weighted by incidents:  {pooled_h:+.1f}%")
print(f"    ATT weighted by agency:     {np.mean(per_h):+.1f}%")
post = d[(d["agency_id"].isin(KEEP)) & (d["period"] == "after")]
shares = post.groupby("agency_id")["n_uof"].sum()
shares = shares / shares.sum()
print(f"    the incident weighted average of the true effects: "
      f"{-100 * sum(effects[a] * shares[a] for a in KEEP):.1f}%")
print(f"    the agency weighted average of the true effects:   "
      f"{-100 * np.mean(list(effects.values())):.1f}%")
print("")
print("    incident shares: "
      + ", ".join(f"{NAME[a].split()[0]} {100 * shares[a]:.0f}%" for a in KEEP))

Now the two estimands separate, and each moves toward its own target: the
incident weighted version toward the large agencies' 20 percent, the agency
weighted version toward the simple average of 12. Neither lands exactly,
because four agencies over 30 months carry real sampling noise, but the
ordering is unambiguous and it is the ordering the weights imply.

Neither is biased. They are estimating different things, and a report that
gives one while answering the other is wrong in a way no diagnostic will
catch.

**With heterogeneous effects, the weighting scheme is part of the estimand
and belongs in its definition.**

## Exercise

The estimand also depends on the **period** it averages over. Compute the ATT
over the first year after full implementation and over the last year.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rows = []
    for label, lo_m, hi_m in [("first 12 settled months", "2023-11", "2024-10"),
                              ("last 12 settled months", "2025-05", "2026-04"),
                              ("all 30 settled months", "2023-11", "2026-04")]:
        s = d[(d["period"] != "after") | ((d["year_month"] >= lo_m)
                                          & (d["year_month"] <= hi_m))]
        e, lo, hi, _ = fit(s, KEEP)
        rows.append({"window": label, "ATT": f"{e:+.1f}%",
                     "95 percent interval": f"[{lo:+.1f}, {hi:+.1f}]"})
    print(f"  the planted effect is constant at {TRUTH:+.1f}% throughout\n")
    display(pd.DataFrame(rows).set_index("window"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

The three windows give similar estimates with different intervals, which is
what a constant effect should produce. The intervals are widest for the
shortest windows, for the reason Intermediate Module 14 sets out.

**The point is that the window is part of the estimand, not a detail of the
analysis.** "The effect of the program" is not defined until you say over
what period. A program with a decaying effect gives very different answers
over the first year and the third, and both are correct ATTs for their
windows.

Stating the window costs six words and removes the most common source of
disagreement between two analysts who both did the work properly.

</details>

---

**Next:** [Module 2: Directed Acyclic Graphs and the Backdoor Criterion](Module_02_DAGs_And_The_Backdoor_Criterion.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*